# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID='task099'; CH=10; H=W=30
ROOT=Path(COMPETITION)
LOCAL_ROOT=Path('/mnt/data')
TASK_PATH=ROOT/f'{TASK_ID}.json'
if not TASK_PATH.exists():
    TASK_PATH=LOCAL_ROOT/f'{TASK_ID}.json'

OUT_DIR=Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
AUDIT_JSON=OUT_DIR/f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/f'{TASK_ID}_zero_pad_audit.csv'
HEIGHTS=[3,4,5]


In [7]:
class Task099CupFillZeroPad(nn.Module):
    def __init__(self):
        super().__init__()
        self.outline=nn.ModuleList(); self.seed=nn.ModuleList(); self.stamp=nn.ModuleList(); self.expected=[]
        for h in HEIGHTS:
            ok=torch.zeros(1,1,h,5)
            ok[0,0,0,0]=1; ok[0,0,0,1]=1; ok[0,0,0,3]=1; ok[0,0,0,4]=1
            ok[0,0,h-1,:]=1
            ok[0,0,1:h-1,0]=1; ok[0,0,1:h-1,4]=1
            conv=nn.Conv2d(1,1,(h,5),bias=False)
            conv.weight.data=ok
            conv.weight.requires_grad=False
            self.outline.append(conv)
            self.expected.append(float(ok.sum()))

            sm=torch.zeros(8,1,h,5)
            sm[:,0,1:h-1,1:4]=1
            sconv=nn.Conv2d(8,8,(h,5),groups=8,bias=False)
            sconv.weight.data=sm
            sconv.weight.requires_grad=False
            self.seed.append(sconv)

            fm=torch.zeros(8,1,h+1,5)
            fm[:,0,0,:]=1        # cap row above the cup
            fm[:,0,1,2]=1        # gap in cup top row
            fm[:,0,2:h,1:4]=1    # interior rows; bottom outline is not overwritten
            deconv=nn.ConvTranspose2d(8,8,(h+1,5),groups=8,bias=False)
            deconv.weight.data=fm
            deconv.weight.requires_grad=False
            self.stamp.append(deconv)

    def forward(self,x):
        # Correct contract: x is one-hot only in the real grid; outside padding is all zeros.
        active=torch.clamp(x.sum(dim=1,keepdim=True),0.0,1.0)
        outline=x[:,1:2]
        seeds=x[:,2:10]
        acc=seeds*0
        for i,h in enumerate(HEIGHTS):
            od=(self.outline[i](outline) > (self.expected[i]-0.5)).to(x.dtype)
            sd=(self.seed[i](seeds) > 0.5).to(x.dtype)
            d=sd*od
            z=d[:,:,:1,:]*0
            dshift=torch.cat([d[:,:,1:,:], z], dim=2)
            acc=acc + self.stamp[i](dshift)[:,:,:30,:30]
        fills=torch.clamp(acc,0,1)
        nonbg=torch.clamp(torch.cat([x[:,1:2], fills],dim=1),0,1) * active
        occ=torch.clamp(nonbg.sum(dim=1,keepdim=True),0,1)
        bg=active*(1-occ)
        return torch.cat([bg,nonbg],dim=1)


In [8]:
def grid_to_tensor(grid):
    # Real ARC rectangle: one-hot including background channel 0.
    # Outside rectangle: all-zero padding.
    a=np.array(grid,dtype=np.int64)
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    hh=min(H,a.shape[0]); ww=min(W,a.shape[1])
    for r in range(hh):
        for c in range(ww):
            v=int(a[r,c])
            if 0<=v<CH:
                x[0,v,r,c]=1.0
    return x

def full_target_grid(grid):
    # For argmax comparison only; raw tensor check below enforces zero padding.
    a=np.array(grid,dtype=np.int64)
    y=np.zeros((H,W),dtype=np.int64)
    y[:min(H,a.shape[0]),:min(W,a.shape[1])]=a[:H,:W]
    return y

def target_zero_onehot(grid):
    a=np.array(grid,dtype=np.int64)
    y=np.zeros((1,CH,H,W),dtype=np.float32)
    hh=min(H,a.shape[0]); ww=min(W,a.shape[1])
    for r in range(hh):
        for c in range(ww):
            v=int(a[r,c])
            if 0<=v<CH:
                y[0,v,r,c]=1.0
    return y

def onnx_shape(v):
    return [d.dim_value for d in v.type.tensor_type.shape.dim]

def op_counts(m):
    out={}
    for n in m.graph.node:
        out[n.op_type]=out.get(n.op_type,0)+1
    return dict(sorted(out.items()))

def exact_eval(sess,cases):
    exact=0; raw_ok_count=0; first_wrong=None
    inp=sess.get_inputs()[0].name
    for i,c in enumerate(cases):
        y=sess.run(None,{inp:grid_to_tensor(c['input'])})[0]
        pred=y[0].argmax(0)
        tgt=full_target_grid(c['output'])
        raw_tgt=target_zero_onehot(c['output'])
        argmax_ok=np.array_equal(pred,tgt)
        raw_ok=np.array_equal(np.round(y,6),raw_tgt)
        exact += int(argmax_ok)
        raw_ok_count += int(raw_ok)
        if (not argmax_ok or not raw_ok) and first_wrong is None:
            a=np.array(c['output']); hh=min(H,a.shape[0]); ww=min(W,a.shape[1])
            sums=y.sum(axis=1)
            first_wrong={
                'idx':i,
                'argmax_ok':bool(argmax_ok),
                'raw_zero_pad_ok':bool(raw_ok),
                'wrong_argmax_pixels':int((pred!=tgt).sum()),
                'unique_values':[float(v) for v in np.unique(np.round(y,6))[:20]],
                'sum_inside_min':float(sums[0,:hh,:ww].min()) if hh and ww else None,
                'sum_inside_max':float(sums[0,:hh,:ww].max()) if hh and ww else None,
                'sum_outside_max':float(max(sums[0,hh:,:].max() if hh<H else 0, sums[0,:,:ww*0].max() if False else 0, sums[0,:,ww:].max() if ww<W else 0)),
            }
    return {'exact':exact,'total':len(cases),'raw_zero_pad':raw_ok_count,'first_wrong':first_wrong}


In [9]:
def make_synthetic_case(height,width,objects):
    grid=np.zeros((height,width),dtype=np.int64)
    valid=[]
    for top,left,h,color in objects:
        if top<1 or top+h>height or left<0 or left+5>width or color in [0,1]:
            continue
        valid.append((top,left,h,color))
        grid[top,left]=1; grid[top,left+1]=1; grid[top,left+3]=1; grid[top,left+4]=1
        grid[top+h-1,left:left+5]=1
        grid[top+1:top+h-1,left]=1; grid[top+1:top+h-1,left+4]=1
        rr=top+1+((top+left+color)%(h-2))
        cc=left+1+((left+color)%3)
        grid[rr,cc]=color
    out=grid.copy()
    for top,left,h,color in valid:
        out[top-1,left:left+5]=color
        out[top,left+2]=color
        out[top+1:top+h-1,left+1:left+4]=color
    return {'input':grid.tolist(),'output':out.tolist()}

def synthetic_suite():
    cases=[]
    for HH,WW in [(10,10),(12,12),(15,20),(20,15),(30,30)]:
        for h in HEIGHTS:
            for top in range(1,max(2,HH-h+1)):
                for left in [0,1,2,max(0,WW-5),max(0,WW-7)]:
                    if left+5<=WW and top+h<=HH:
                        cases.append(make_synthetic_case(HH,WW,[(top,left,h,2+((top+left+h)%8))]))
    for h1 in HEIGHTS:
        for h2 in HEIGHTS:
            cases.append(make_synthetic_case(30,30,[(1,0,h1,2+h1),(12,10,h2,9-h2)]))
            cases.append(make_synthetic_case(30,30,[(5,20,h1,9),(18,2,h2,3)]))
    return cases


In [10]:
with open(TASK_PATH) as f:
    task=json.load(f)

model=Task099CupFillZeroPad().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
dummy[:,0,:,:]=1

torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],
                  opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)
ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
risk={'ScatterND','Shape','Range','Expand','Gather','ConstantOfShape'}
tree={op for op in ops if 'Tree' in op}

sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
ag=task.get('arc-gen',[])
hold=ag[len(ag)-int(math.ceil(0.6*len(ag))):]
syn=synthetic_suite()
health={
    'task':TASK_ID,
    'method':'zero-padding grouped convolutional cup detector plus grouped transposed-convolution painter',
    'input_shape':onnx_shape(m.graph.input[0]),
    'output_shape':onnx_shape(m.graph.output[0]),
    'size_bytes':ONNX_PATH.stat().st_size,
    'op_counts':ops,
    'forbidden_ops_present':sorted(forbidden.intersection(ops)),
    'risk_ops_present':sorted(risk.intersection(ops)),
    'tree_ops_present':sorted(tree),
    'accuracy':{
        'train':exact_eval(sess,task.get('train',[])),
        'test':exact_eval(sess,task.get('test',[])),
        'arc-gen':exact_eval(sess,ag),
        'arc-gen-holdout-60pct':exact_eval(sess,hold),
        'synthetic-ood':exact_eval(sess,syn),
    },
    'synthetic_count':len(syn),
}

assert health['input_shape']==[1,10,30,30]
assert health['output_shape']==[1,10,30,30]
assert health['size_bytes']<1_400_000
assert not health['forbidden_ops_present']
assert not health['risk_ops_present']
assert not health['tree_ops_present']
for k,v in health['accuracy'].items():
    assert v['exact']==v['total'], (k,v)
    assert v['raw_zero_pad']==v['total'], (k,v)

with open(AUDIT_JSON,'w') as f: json.dump(health,f,indent=2)
with open(AUDIT_CSV,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['key','value'])
    for k,v in health.items():
        w.writerow([k,json.dumps(v) if not isinstance(v,(str,int,float)) else v])
health


/tmp/ipykernel_16/2020329892.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],


{'task': 'task099',
 'method': 'zero-padding grouped convolutional cup detector plus grouped transposed-convolution painter',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 17296,
 'op_counts': {'Add': 3,
  'Cast': 6,
  'Clip': 4,
  'Concat': 5,
  'Constant': 76,
  'Conv': 6,
  'ConvTranspose': 3,
  'Greater': 6,
  'Mul': 9,
  'ReduceSum': 2,
  'Slice': 14,
  'Sub': 1},
 'forbidden_ops_present': [],
 'risk_ops_present': [],
 'tree_ops_present': [],
 'accuracy': {'train': {'exact': 3,
   'total': 3,
   'raw_zero_pad': 3,
   'first_wrong': None},
  'test': {'exact': 1, 'total': 1, 'raw_zero_pad': 1, 'first_wrong': None},
  'arc-gen': {'exact': 261,
   'total': 261,
   'raw_zero_pad': 261,
   'first_wrong': None},
  'arc-gen-holdout-60pct': {'exact': 157,
   'total': 157,
   'raw_zero_pad': 157,
   'first_wrong': None},
  'synthetic-ood': {'exact': 1023,
   'total': 1023,
   'raw_zero_pad': 1023,
   'first_wrong': None}},
 'synthetic_count': 1023}

In [11]:
# package zip with root task099.onnx. GENERIC_ZIP is /kaggle/working/submission.zip.
for zp in [ZIP_PATH, GENERIC_ZIP]:
    if zp.exists():
        zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
ZIP_PATH, GENERIC_ZIP


(PosixPath('/kaggle/working/task099_zero_pad_static_graph_submission.zip'),
 PosixPath('/kaggle/working/submission.zip'))